In [ ]:
!pip install matplotlib numpy

In [ ]:
import sys
import time
import matplotlib as plt
import numpy as np

# Aumenta o limite de recursão nativo do Python para suportar Ackermann
STACK_LIMIT=10000
sys.setrecursionlimit(STACK_LIMIT)

# MÓDULO 1: FUNÇÕES BÁSICAS INICIAIS

In [ ]:
def Z(x: int) -> int:
    """Função Zero: Z(x) = 0"""
    return 0

def S(x: int) -> int:
    """Função Sucessor: S(x) = x + 1"""
    return x + 1

def P(i: int, n: int, *args) -> int:
    """
    Função Projeção: P_i^n(x_1, ..., x_n) = x_i
    Nota: 'i' usa indexação baseada em 1 (1-indexed) de acordo com a teoria.
    """
    if len(args) != n:
        raise ValueError(f"Esperado {n} argumentos, recebido {len(args)}")
    if i < 1 or i > n:
        raise IndexError(f"Índice i={i} fora dos limites para n={n}")
    return args[i - 1]

# Testes de verificação dos primitivos
print("--- Testes das Funções Básicas ---")
print(f"Z(10) = {Z(10)}")
print(f"S(5) = {S(5)}")
print(f"P_2^3(10, 20, 30) = {P(2, 3, 10, 20, 30)}")

# MÓDULO 2: FUNÇÕES RECURSIVAS PRIMITIVAS (FRP)


In [ ]:
def add_frp(x: int, y: int) -> int:
    """
    Adição definida via FRP:
    add(x, 0) = x
    add(x, y + 1) = S(add(x, y))
    """
    result = x  # Caso base: add(x, 0)
    for _ in range(y):  # Passos de recursão
        result = S(result)
    return result

def mult_frp(x: int, y: int) -> int:
    """
    Multiplicação definida via FRP:
    mult(x, 0) = 0
    mult(x, y + 1) = add(mult(x, y), x)
    """
    result = Z(x)  # Caso base: mult(x, 0)
    for _ in range(y):
        result = add_frp(result, x)
    return result

# Testes das operações FRP
print("--- Testes das Funções Recursivas Primitivas ---")
print(f"add_frp(7, 8) = {add_frp(7, 8)}")
print(f"mult_frp(4, 5) = {mult_frp(4, 5)}")

# MÓDULO 3: ACKERMANN & OPERADOR DE MINIMIZAÇÃO (μ)


In [ ]:
def ackermann(m: int, n: int) -> int:
    """
    Função de Ackermann-Péter:
    A(0, n) = n + 1
    A(m, 0) = A(m - 1, 1)
    A(m, n) = A(m - 1, A(m, n - 1))
    """
    if m == 0:
        return n + 1
    elif n == 0:
        return ackermann(m - 1, 1)
    else:
        return ackermann(m - 1, ackermann(m, n - 1))

def mu_operator(f, target=0):
    """
    Operador de Minimização Não Limitado (μ):
    Retorna o menor y ∈ ℕ tal que f(y) == target.
    Equivale a um laço while indeterminado.
    """
    y = 0
    while True:
        if f(y) == target:
            return y
        y = S(y)

# Testes de Ackermann e Minimização
print("--- Testes da Função de Ackermann ---")
print(f"A(1, 2) = {ackermann(1, 2)}")
print(f"A(2, 2) = {ackermann(2, 2)}")
print(f"A(3, 3) = {ackermann(3, 3)}")
print(f"A(3, 4) = {ackermann(3, 4)}")
# Exemplo do operador mu: busca pela raiz exata (y^2 - x = 0)
x_test = 49
raiz_exata = mu_operator(lambda y: mult_frp(y, y) - x_test)
print(f"\n--- Teste do Operador μ ---")
print(f"Menor y tal que y^2 = {x_test} -> y = {raiz_exata}")

# MÓDULO 4: RAIZ QUADRADA (FRP vs MINIMIZAÇÃO)


In [ ]:
def sqrt_frp_bounded(x: int) -> int:
    """
    Raiz Quadrada de Piso (FRP):
    Garante parada limitando a busca até x (limite superior razoável).
    Retorna o maior y tal que y^2 <= x.
    """
    ans = 0
    # Como y <= x para todo x >= 0, limitamos o loop finitamente até x+1
    for y in range(x + 1):
        if mult_frp(y, y) <= x:
            ans = y
        else:
            break
    return ans

def sqrt_mu_unbounded(x: int) -> int:
    """
    Raiz Quadrada Inteira Exata / Busca Livre (μ-Recursiva):
    Encontra o menor y tal que (y+1)^2 > x usando busca indeterminada (while).
    """
    def pred(y):
        # f(y) = 0 quando (y+1)^2 > x
        return 0 if mult_frp(S(y), S(y)) > x else 1

    return mu_operator(pred, target=0)

# Testes comparativos
test_val = 20
print(f"Raiz de piso (FRP Bounded) de {test_val}: {sqrt_frp_bounded(test_val)}")
print(f"Raiz de piso (μ Unbounded) de {test_val}: {sqrt_mu_unbounded(test_val)}")

# MÓDULO 5: BENCHMARK & ANÁLISE DE DESEMPENHO


In [ ]:
def benchmark():
    print("==================================================")
    print("          BENCHMARK DE DESEMPENHO               ")
    print("==================================================\n")

    # 1. Benchmark de Ackermann
    print("[1] Testando Limites de Crescimento da Função de Ackermann:")
    ack_inputs = [(0, 0), (1, 2), (2, 2), (3, 3), (3, 4)]
    for m, n in ack_inputs:
        start = time.perf_counter()
        res = ackermann(m, n)
        elapsed = time.perf_counter() - start
        print(f"  A({m}, {n}) = {res:<10} | Tempo: {elapsed:.6f} segundos")

    # 2. Benchmark FRP vs μ (Raiz Quadrada)
    print("\n[2] Comparando Busca FRP (Limitada) vs Busca μ (Minimização):")
    numeros = [100, 1000, 5000]

    for n in numeros:
        # FRP
        t0 = time.perf_counter()
        r_frp = sqrt_frp_bounded(n)
        t_frp = time.perf_counter() - t0

        # μ-Operator
        t0 = time.perf_counter()
        r_mu = sqrt_mu_unbounded(n)
        t_mu = time.perf_counter() - t0

        print(f"  Número: {n:<5}")
        print(f"    - FRP Bounded  : resultado={r_frp}, tempo={t_frp:.6f}s")
        print(f"    - μ Unbounded  : resultado={r_mu}, tempo={t_mu:.6f}s")

benchmark()

# MÓDULO 6: GRÁFICO DE CONSUMO DA PILHA (ACKERMANN)
